In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

/home/adellorto/Projects/MiniGPT/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [1]:
from tokenizer import CharachterLevelTokenizer, TiktokenTokenizer, MinbpeTokenizer

In [2]:
# read it in to inspect it
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
char_tokenizer = CharachterLevelTokenizer(text)
tiktoken_tokenizer = TiktokenTokenizer(text)
minbpe_tokenizer = MinbpeTokenizer(text)

print(f"CharTokenizer vocab size:       {len(char_tokenizer.vocab)}")
print(f"Tiktoken compact vocab size:    {len(tiktoken_tokenizer.vocab)}")
print(f"Tiktoken full vocab size:       {tiktoken_tokenizer.encoding.n_vocab}")
print(f"MinBPE compact vocab size:      {len(minbpe_tokenizer.vocab)}")

CharTokenizer vocab size:       65
Tiktoken compact vocab size:    12111
Tiktoken full vocab size:       100277
MinBPE compact vocab size:      109


In [ ]:
base_params = {
    'batch_size' : 32,
    'block_size' : 8,
    'max_iters' : 3000,
    'eval_interval' : 300,
    'learning_rate' : 1e-2,
    'eval_iters' : 200, 
    'n_embd' : 64,
    'n_head' : 2,
    'n_layer': 3,   
    'dropout' : 0.1
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)

In [8]:
def get_batch(data, params, device):
    ix = torch.randint(len(data) - params['block_size'], (params['batch_size'],))
    x = torch.stack([data[i:i+params['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+params['block_size']+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
@torch.no_grad()
def estimate_loss(model, data, params, device):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(params['eval_iters'])
        for k in range(params['eval_iters']):
            X, Y = get_batch(data[split], params, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, params):
        super().__init__()
        self.key = nn.Linear(params['n_embd'], head_size, bias=False)
        self.query = nn.Linear(params['n_embd'], head_size, bias=False)
        self.value = nn.Linear(params['n_embd'], head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(params['block_size'], params['block_size'])))

        self.dropout = nn.Dropout(params['dropout'])

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, params, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, params) for _ in range(params['n_head'])])
        self.proj = nn.Linear(head_size * params['n_head'], params['n_embd'])
        self.dropout = nn.Dropout(params['dropout'])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, params):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(params['n_embd'], 4 * params['n_embd']),
            nn.ReLU(),
            nn.Linear(4 * params['n_embd'], params['n_embd']),
            nn.Dropout(params['dropout']),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, params):
        super().__init__()
        head_size = params['n_embd'] // params['n_head']
        self.sa = MultiHeadAttention(params, head_size)
        self.ffwd = FeedFoward(params)
        self.ln1 = nn.LayerNorm(params['n_embd'])
        self.ln2 = nn.LayerNorm(params['n_embd'])

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self, params):
        super().__init__()
        self.params = params
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(params['vocab_size'], params['n_embd'])
        self.position_embedding_table = nn.Embedding(params['block_size'], params['n_embd'])
        self.blocks = nn.Sequential(*[Block(params) for _ in range(params['n_layer'])])
        self.ln_f = nn.LayerNorm(params['n_embd']) # final layer norm
        self.lm_head = nn.Linear(params['n_embd'], params['vocab_size'])

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -self.params['block_size']:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
from tqdm import tqdm

def run_experiment(tokenizer, name):
    # Encode full text first
    encoded = torch.tensor(tokenizer.encode(text), dtype=torch.long)

    vocab_size = len(tokenizer.vocab)
    params = {**base_params, 'vocab_size': vocab_size}

    # Train/val split
    n = int(0.9 * len(encoded))
    data = {'train': encoded[:n], 'val': encoded[n:]}

    # Model + optimizer
    torch.manual_seed(42)
    model = GPTLanguageModel(params).to(device)
    print(f"\n=== {name} | vocab_size={vocab_size} | tokens={len(encoded)} | {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters ===")
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'])

    history = []
    pbar = tqdm(range(params['max_iters'] + 1), desc=name)
    for iter in pbar:
        if iter % params['eval_interval'] == 0:
            losses = estimate_loss(model, data, params, device)
            pbar.set_postfix(train=f"{losses['train']:.4f}", val=f"{losses['val']:.4f}")
            tqdm.write(f"  step {iter:4d}: train {losses['train']:.4f}  val {losses['val']:.4f}")
            history.append({'step': iter, 'train': losses['train'].item(), 'val': losses['val'].item()})
        if iter < params['max_iters']:
            xb, yb = get_batch(data['train'], params, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

    return model, history

In [ ]:
char_model, char_history = run_experiment(char_tokenizer, "CharacterLevel")

In [ ]:
tiktoken_model, tiktoken_history = run_experiment(tiktoken_tokenizer, "Tiktoken (cl100k)")

In [ ]:
minbpe_model, minbpe_history = run_experiment(minbpe_tokenizer, "MinBPE")

In [ ]:
import os

save_dir = 'models/attention'
os.makedirs(save_dir, exist_ok=True)

for model, name in zip([char_model, tiktoken_model, minbpe_model], ['char', 'tiktoken', 'minbpe']):
    torch.save(model.state_dict(), os.path.join(save_dir, f'{name}_model.pt'))

print(f"Models saved to {save_dir}/")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)

for ax, history, name in zip(axes, [char_history, tiktoken_history, minbpe_history], ["CharacterLevel", "Tiktoken (cl100k)", "MinBPE"]):
    steps = [h['step'] for h in history]
    ax.plot(steps, [h['train'] for h in history], label='train')
    ax.plot(steps, [h['val'] for h in history], label='val', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.legend()

plt.suptitle('GPT Model: Loss Comparison by Tokenizer')
plt.tight_layout()
plt.show()

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("=== CharacterLevel generation ===")
tokens = char_model.generate(context, max_new_tokens=500)[0].tolist()
print(char_tokenizer.decode(tokens))

print("\n=== Tiktoken generation ===")
tokens = tiktoken_model.generate(context, max_new_tokens=500)[0].tolist()
print(tiktoken_tokenizer.decode(tokens))

print("\n=== MinBPE generation ===")
tokens = minbpe_model.generate(context, max_new_tokens=500)[0].tolist()
print(minbpe_tokenizer.decode(tokens))